In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 1000

data = {
    "duration": np.random.randint(1, 1000, n),
    "protocol": np.random.choice(["TCP", "UDP", "ICMP"], n),
    "src_bytes": np.random.randint(100, 100000, n),
    "dst_bytes": np.random.randint(100, 100000, n),
    "failed_logins": np.random.randint(0, 10, n),
    "connection_count": np.random.randint(1, 100, n),
    "is_suspicious": np.random.choice([0, 1], n, p=[0.8, 0.2])
}

df = pd.DataFrame(data)

df.head()

,duration,protocol,src_bytes,dst_bytes,failed_logins,connection_count,is_suspicious
0,103,TCP,51856,42700,6,28,1
1,436,TCP,97724,59660,3,11,0
2,861,ICMP,26392,78140,2,78,0
3,271,TCP,86306,61658,0,39,0
4,107,UDP,69374,86305,7,47,1


In [2]:
print("Dataset Shape:", df.shape)
print("\nDataset Information:")
df.info()

print("\nThreat Distribution:")
print(df["is_suspicious"].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (1000, 7)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   duration          1000 non-null   int32
 1   protocol          1000 non-null   str  
 2   src_bytes         1000 non-null   int32
 3   dst_bytes         1000 non-null   int32
 4   failed_logins     1000 non-null   int32
 5   connection_count  1000 non-null   int32
 6   is_suspicious     1000 non-null   int64
dtypes: int32(5), int64(1), str(1)
memory usage: 38.5 KB

Threat Distribution:
is_suspicious
0    802
1    198
Name: count, dtype: int64

Missing Values:
duration            0
protocol            0
src_bytes           0
dst_bytes           0
failed_logins       0
connection_count    0
is_suspicious       0
dtype: int64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

df_model = df.copy()

encoder = LabelEncoder()
df_model["protocol"] = encoder.fit_transform(df_model["protocol"])

X = df_model.drop("is_suspicious", axis=1)
y = df_model["is_suspicious"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("CyberShield AI - Threat Detection Results")
print("-----------------------------------------")
print("Accuracy:", accuracy_score(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

CyberShield AI - Threat Detection Results
-----------------------------------------
Accuracy: 0.785

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.93      0.87       160
           1       0.43      0.23      0.30        40

    accuracy                           0.79       200
   macro avg       0.63      0.58      0.58       200
weighted avg       0.75      0.79      0.76       200


Confusion Matrix:
[[148  12]
 [ 31   9]]


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42
    )
}

results = []

for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1 Score": f1_score(y_test, pred, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.580,0.250000,0.550,0.343750
1,Decision Tree,0.680,0.227273,0.250,0.238095
2,Random Forest,0.785,0.428571,0.225,0.295082


In [5]:
df["is_suspicious"] = (
    (df["failed_logins"] >= 6) |
    (df["connection_count"] >= 80) |
    ((df["src_bytes"] > 80000) & (df["dst_bytes"] < 500))
).astype(int)

print("Updated Threat Distribution:")
print(df["is_suspicious"].value_counts())

print("\nThreat Percentage:")
print(round(df["is_suspicious"].mean() * 100, 2), "%")

Updated Threat Distribution:
is_suspicious
1    505
0    495
Name: count, dtype: int64

Threat Percentage:
50.5 %


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df_model = df.copy()

encoder = LabelEncoder()
df_model["protocol"] = encoder.fit_transform(df_model["protocol"])

X = df_model.drop("is_suspicious", axis=1)
y = df_model["is_suspicious"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
}

results = []

for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred), 3),
        "Precision": round(precision_score(y_test, pred, zero_division=0), 3),
        "Recall": round(recall_score(y_test, pred, zero_division=0), 3),
        "F1 Score": round(f1_score(y_test, pred, zero_division=0), 3)
    })

results_df = pd.DataFrame(results)
results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.805,0.792,0.832,0.812
1,Decision Tree,1.000,1.000,1.000,1.000
2,Random Forest,1.000,1.000,1.000,1.000


In [7]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

best_model = models["Random Forest"]

joblib.dump(best_model, "../models/cybershield_model.pkl")
joblib.dump(encoder, "../models/protocol_encoder.pkl")

print("CyberShield AI model saved successfully!")

CyberShield AI model saved successfully!


In [8]:
%pip install transformers torch


Note: you may need to restart the kernel to use updated packages.


In [9]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face integration ready!")

C:\Users\aashi\anaconda3\envs\cybershield\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.15.0+cpu
Transformers: 5.24.0
Hugging Face integration ready!


In [10]:
from transformers import pipeline

analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1
)

print("CyberShield AI NLP analyzer loaded!")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 649.97it/s]


CyberShield AI NLP analyzer loaded!


In [11]:
messages = [
    "Multiple failed login attempts detected from one IP address.",
    "Normal user login successful.",
    "Unusual network traffic detected on port 22."
]

for message in messages:
    result = analyzer(message)
    print("Message:", message)
    print("NLP result:", result)
    print("-" * 50)

Message: Multiple failed login attempts detected from one IP address.
NLP result: [{'label': 'NEGATIVE', 'score': 0.9996309280395508}]
--------------------------------------------------
Message: Normal user login successful.
NLP result: [{'label': 'POSITIVE', 'score': 0.9992042183876038}]
--------------------------------------------------
Message: Unusual network traffic detected on port 22.
NLP result: [{'label': 'NEGATIVE', 'score': 0.9792892932891846}]
--------------------------------------------------


In [12]:
from transformers import pipeline

security_classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-1",
    device=-1
)

print("Cybersecurity NLP classifier loaded!")

Loading weights: 100%|██████████| 231/231 [00:00<00:00, 9653.70it/s]


Cybersecurity NLP classifier loaded!


In [13]:
labels = [
    "Brute force attack",
    "Port scanning",
    "Denial of service attack",
    "Normal network activity"
]

messages = [
    "Multiple failed login attempts detected from one IP address.",
    "An IP address is scanning hundreds of network ports.",
    "A server is receiving an unusually large number of requests."
]

for message in messages:
    result = security_classifier(
        message,
        candidate_labels=labels
    )

    print("\nEvent:", message)
    print("Predicted category:", result["labels"][0])
    print("Confidence:", round(result["scores"][0] * 100, 2), "%")


Event: Multiple failed login attempts detected from one IP address.
Predicted category: Port scanning
Confidence: 42.25 %

Event: An IP address is scanning hundreds of network ports.
Predicted category: Port scanning
Confidence: 95.25 %

Event: A server is receiving an unusually large number of requests.
Predicted category: Denial of service attack
Confidence: 67.31 %


In [19]:
security_events = [
    "A single source IP repeatedly attempts to log in using different passwords. The authentication system records many failed login attempts. This indicates a brute-force login attack.",

    "A remote IP address sends connection requests to hundreds of different TCP and UDP ports on a target server to discover open services. This indicates network port scanning.",

    "A server receives an extremely high volume of incoming requests from multiple sources, exhausting its resources and preventing legitimate users from accessing the service. This indicates a denial-of-service attack.",

    "A regular employee successfully logs in using valid credentials and accesses an authorized application. No unusual network activity is observed."
]

threat_labels = [
    "a brute-force attack using repeated failed login attempts to guess passwords",
    "network port scanning to discover open ports on a server",
    "a denial-of-service attack overwhelming a server with excessive requests",
    "normal network activity with legitimate user login behavior"
]


for event in security_events:
    result = security_classifier(
        event,
        candidate_labels=threat_labels,
        hypothesis_template="This cybersecurity event is an example of {}."
    )

    print("\nEvent:", event)
    print("Prediction:", result["labels"][0])
    print("Score:", round(result["scores"][0] * 100, 2), "%")
    print("-" * 60)


Event: A single source IP repeatedly attempts to log in using different passwords. The authentication system records many failed login attempts. This indicates a brute-force login attack.
Prediction: a brute-force attack using repeated failed login attempts to guess passwords
Score: 97.43 %
------------------------------------------------------------

Event: A remote IP address sends connection requests to hundreds of different TCP and UDP ports on a target server to discover open services. This indicates network port scanning.
Prediction: network port scanning to discover open ports on a server
Score: 98.9 %
------------------------------------------------------------

Event: A server receives an extremely high volume of incoming requests from multiple sources, exhausting its resources and preventing legitimate users from accessing the service. This indicates a denial-of-service attack.
Prediction: a denial-of-service attack overwhelming a server with excessive requests
Score: 99.32 

In [23]:
def cybershield_analyze(event, features):

    ml_prediction = best_model.predict(
        pd.DataFrame([features], columns=X.columns)
    )[0]

    nlp_result = security_classifier(
        event,
        candidate_labels=threat_labels,
        hypothesis_template="This cybersecurity event is an example of {}."
    )

    return {
        "ML Prediction": "Suspicious" if ml_prediction == 1 else "Normal",
        "NLP Prediction": nlp_result["labels"][0],
        "NLP Score": round(nlp_result["scores"][0] * 100, 2)
    }

In [24]:
test_event = "Multiple failed login attempts detected from one IP address."

test_features = {
    "duration": 250,
    "protocol": encoder.transform(["TCP"])[0],
    "src_bytes": 500,
    "dst_bytes": 200,
    "failed_logins": 9,
    "connection_count": 15
}

result = cybershield_analyze(test_event, test_features)

print("CYBERSHIELD AI - SECURITY ANALYSIS")
print("----------------------------------")

for key, value in result.items():
    print(f"{key}: {value}")

CYBERSHIELD AI - SECURITY ANALYSIS
----------------------------------
ML Prediction: Suspicious
NLP Prediction: a brute-force attack using repeated failed login attempts to guess passwords
NLP Score: 64.51


In [22]:
test_events = [
    "Multiple failed login attempts detected from one IP address.",
    "An attacker is checking many ports on a server.",
    "A large volume of requests is overwhelming the web server.",
    "A user successfully logged in with normal activity."
]

for event in test_events:
    result = security_classifier(
        event,
        candidate_labels=threat_labels,
        hypothesis_template="This cybersecurity event is an example of {}."
    )

    print("\nEvent:", event)
    print("Prediction:", result["labels"][0])
    print("Score:", round(result["scores"][0] * 100, 2), "%")


Event: Multiple failed login attempts detected from one IP address.
Prediction: a brute-force attack using repeated failed login attempts to guess passwords
Score: 64.51 %

Event: An attacker is checking many ports on a server.
Prediction: network port scanning to discover open ports on a server
Score: 88.31 %

Event: A large volume of requests is overwhelming the web server.
Prediction: a denial-of-service attack overwhelming a server with excessive requests
Score: 90.19 %

Event: A user successfully logged in with normal activity.
Prediction: normal network activity with legitimate user login behavior
Score: 98.69 %


In [25]:
%pip install sentence-transformers faiss-cpu

  Using cached sentence_transformers-6.3.0-py3-none-any.whl.metadata (20 kB)
  Using cached faiss_cpu-1.16.0-cp311-cp311-win_amd64.whl.metadata (7.8 kB)
Using cached sentence_transformers-6.3.0-py3-none-any.whl (780 kB)
Using cached faiss_cpu-1.16.0-cp311-cp311-win_amd64.whl (16.6 MB)

   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transformers]
   -------------------- ------------------- 1/2 [sentence-transfo

In [26]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("RAG libraries loaded successfully!")

RAG libraries loaded successfully!


In [27]:
security_docs = [
    "Brute-force attacks involve repeated login attempts to guess a user's password. Multiple failed logins from one IP address may indicate this activity.",

    "Port scanning is a technique used to identify open ports and services on a network. Unusual connection attempts across many ports may indicate reconnaissance.",

    "Denial-of-service attacks overwhelm a server with excessive requests, making services unavailable to legitimate users.",

    "Normal network activity includes legitimate user logins, expected data transfers, and authorized service communication.",

    "Repeated authentication failures should be investigated using timestamps, source IP addresses, account activity, and authentication logs.",

    "Network monitoring can identify unusual connection counts, unexpected traffic patterns, and repeated connection attempts.",

    "Security alerts are indicators for investigation, not proof of malicious activity. Analysts should verify events using additional evidence."
]

print("Knowledge base created:", len(security_docs), "documents")

Knowledge base created: 7 documents


In [28]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

document_embeddings = embedding_model.encode(
    security_docs,
    convert_to_numpy=True
)

print("Embeddings generated successfully!")
print("Embedding shape:", document_embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3029.23it/s]


Embeddings generated successfully!
Embedding shape: (7, 384)


In [29]:
dimension = document_embeddings.shape[1]

vector_db = faiss.IndexFlatL2(dimension)

vector_db.add(
    document_embeddings.astype("float32")
)

print("FAISS vector database created!")
print("Total indexed documents:", vector_db.ntotal)

FAISS vector database created!
Total indexed documents: 7


In [30]:
def retrieve_security_knowledge(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = vector_db.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for index, distance in zip(indices[0], distances[0]):
        if index != -1:
            results.append({
                "document": security_docs[index],
                "distance": round(float(distance), 4)
            })

    return results

In [31]:
query = "What does repeated failed login attempts indicate?"

results = retrieve_security_knowledge(query)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Knowledge:", result["document"])
    print("Distance:", result["distance"])


Result 1
Knowledge: Brute-force attacks involve repeated login attempts to guess a user's password. Multiple failed logins from one IP address may indicate this activity.
Distance: 0.4881

Result 2
Knowledge: Repeated authentication failures should be investigated using timestamps, source IP addresses, account activity, and authentication logs.
Distance: 0.5522

Result 3
Knowledge: Network monitoring can identify unusual connection counts, unexpected traffic patterns, and repeated connection attempts.
Distance: 1.1273


In [32]:
def cyber_intelligence(event):

    knowledge = retrieve_security_knowledge(event, top_k=3)

    classification = security_classifier(
        event,
        candidate_labels=threat_labels,
        hypothesis_template="This cybersecurity event is an example of {}."
    )

    report = {
        "Security Event": event,
        "Detected Category": classification["labels"][0],
        "Model Score": round(classification["scores"][0] * 100, 2),
        "Retrieved Intelligence": [
            item["document"] for item in knowledge
        ]
    }

    return report

In [33]:
report = cyber_intelligence(
    "Multiple failed login attempts detected from one IP address."
)

print("CYBERSHIELD AI - THREAT INTELLIGENCE")
print("-----------------------------------")

for key, value in report.items():
    print(f"\n{key}:")
    print(value)

CYBERSHIELD AI - THREAT INTELLIGENCE
-----------------------------------

Security Event:
Multiple failed login attempts detected from one IP address.

Detected Category:
a brute-force attack using repeated failed login attempts to guess passwords

Model Score:
64.51

Retrieved Intelligence:
["Brute-force attacks involve repeated login attempts to guess a user's password. Multiple failed logins from one IP address may indicate this activity.", 'Repeated authentication failures should be investigated using timestamps, source IP addresses, account activity, and authentication logs.', 'Network monitoring can identify unusual connection counts, unexpected traffic patterns, and repeated connection attempts.']


In [34]:
def cybershield_full_analysis(event, features):

    ml_prediction = best_model.predict(
        pd.DataFrame([features], columns=X.columns)
    )[0]

    nlp_result = security_classifier(
        event,
        candidate_labels=threat_labels,
        hypothesis_template="This cybersecurity event is an example of {}."
    )

    rag_results = retrieve_security_knowledge(event, top_k=3)

    return {
        "ML Detection": "Suspicious" if ml_prediction == 1 else "Normal",
        "NLP Classification": nlp_result["labels"][0],
        "NLP Score": round(nlp_result["scores"][0] * 100, 2),
        "Security Intelligence": [
            item["document"] for item in rag_results
        ]
    }

In [35]:
event = "Multiple failed login attempts detected from one IP address."

features = {
    "duration": 250,
    "protocol": encoder.transform(["TCP"])[0],
    "src_bytes": 500,
    "dst_bytes": 200,
    "failed_logins": 9,
    "connection_count": 15
}

analysis = cybershield_full_analysis(event, features)

print("CYBERSHIELD AI - FULL SECURITY ANALYSIS")
print("=" * 45)

for key, value in analysis.items():
    print(f"\n{key}:")
    print(value)

CYBERSHIELD AI - FULL SECURITY ANALYSIS

ML Detection:
Suspicious

NLP Classification:
a brute-force attack using repeated failed login attempts to guess passwords

NLP Score:
64.51

Security Intelligence:
["Brute-force attacks involve repeated login attempts to guess a user's password. Multiple failed logins from one IP address may indicate this activity.", 'Repeated authentication failures should be investigated using timestamps, source IP addresses, account activity, and authentication logs.', 'Network monitoring can identify unusual connection counts, unexpected traffic patterns, and repeated connection attempts.']


In [36]:
import time

security_event_stream = [
    {
        "event": "Multiple failed login attempts detected from one IP address.",
        "features": {
            "duration": 250,
            "protocol": encoder.transform(["TCP"])[0],
            "src_bytes": 500,
            "dst_bytes": 200,
            "failed_logins": 9,
            "connection_count": 15
        }
    },
    {
        "event": "An attacker is checking many ports on a server.",
        "features": {
            "duration": 800,
            "protocol": encoder.transform(["TCP"])[0],
            "src_bytes": 500,
            "dst_bytes": 300,
            "failed_logins": 0,
            "connection_count": 95
        }
    }
]

print("Security event stream initialized!")
print("Total events:", len(security_event_stream))

Security event stream initialized!
Total events: 2


In [37]:
for i, security_event in enumerate(security_event_stream, 1):

    print("\n" + "=" * 50)
    print(f"CYBERSHIELD AI - EVENT {i}")
    print("=" * 50)

    analysis = cybershield_full_analysis(
        security_event["event"],
        security_event["features"]
    )

    print("Event:", security_event["event"])
    print("ML Detection:", analysis["ML Detection"])
    print("NLP Classification:", analysis["NLP Classification"])
    print("NLP Score:", analysis["NLP Score"], "%")

    print("\nRetrieved Security Intelligence:")

    for knowledge in analysis["Security Intelligence"]:
        print("-", knowledge)

    time.sleep(1)


CYBERSHIELD AI - EVENT 1
Event: Multiple failed login attempts detected from one IP address.
ML Detection: Suspicious
NLP Classification: a brute-force attack using repeated failed login attempts to guess passwords
NLP Score: 64.51 %

Retrieved Security Intelligence:
- Brute-force attacks involve repeated login attempts to guess a user's password. Multiple failed logins from one IP address may indicate this activity.
- Repeated authentication failures should be investigated using timestamps, source IP addresses, account activity, and authentication logs.
- Network monitoring can identify unusual connection counts, unexpected traffic patterns, and repeated connection attempts.

CYBERSHIELD AI - EVENT 2
Event: An attacker is checking many ports on a server.
ML Detection: Suspicious
NLP Classification: network port scanning to discover open ports on a server
NLP Score: 88.31 %

Retrieved Security Intelligence:
- Port scanning is a technique used to identify open ports and services on a n

In [38]:
import os

monitoring_code = '''
import time

def monitor_events(event_stream, analysis_function):
    for i, event in enumerate(event_stream, 1):
        print("\\\\n" + "=" * 45)
        print(f"CYBERSHIELD AI - EVENT {i}")
        print("=" * 45)

        result = analysis_function(
            event["event"],
            event["features"]
        )

        for key, value in result.items():
            print(f"\\\\n{key}:")
            print(value)

        time.sleep(1)
'''

os.makedirs("../src/streaming", exist_ok=True)

with open("../src/streaming/monitor.py", "w") as file:
    file.write(monitoring_code)

print("Monitoring module saved successfully!")

Monitoring module saved successfully!


In [39]:
api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(
    title="CyberShield AI",
    description="AI-powered cybersecurity threat analysis API",
    version="1.0"
)

class SecurityEvent(BaseModel):
    event: str
    duration: int
    protocol: str
    src_bytes: int
    dst_bytes: int
    failed_logins: int
    connection_count: int

@app.get("/")
def home():
    return {
        "project": "CyberShield AI",
        "status": "API is running"
    }
'''

os.makedirs("../app/api", exist_ok=True)

with open("../app/api/main.py", "w") as file:
    file.write(api_code)

print("FastAPI backend file created!")

FastAPI backend file created!


In [40]:
api_code = '''
import sys
from pathlib import Path

from fastapi import FastAPI
from pydantic import BaseModel

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

app = FastAPI(
    title="CyberShield AI",
    description="AI-powered cybersecurity threat analysis API",
    version="1.0"
)

class SecurityEvent(BaseModel):
    event: str
    duration: int
    protocol: str
    src_bytes: int
    dst_bytes: int
    failed_logins: int
    connection_count: int

@app.get("/")
def home():
    return {
        "project": "CyberShield AI",
        "status": "API is running"
    }

@app.post("/analyze")
def analyze_event(data: SecurityEvent):
    return {
        "message": "Event received successfully",
        "event": data.event,
        "status": "Ready for AI analysis integration"
    }
'''

with open("../app/api/main.py", "w") as file:
    file.write(api_code)

print("CyberShield AI API updated!")

CyberShield AI API updated!


In [41]:
import os

os.makedirs("../src/detection", exist_ok=True)

inference_code = '''
import joblib
import pandas as pd
import faiss
from pathlib import Path
from transformers import pipeline
from sentence_transformers import SentenceTransformer

ROOT = Path(__file__).resolve().parents[2]
MODEL_DIR = ROOT / "models"

best_model = joblib.load(MODEL_DIR / "cybershield_model.pkl")
encoder = joblib.load(MODEL_DIR / "protocol_encoder.pkl")

feature_columns = [
    "duration", "protocol", "src_bytes",
    "dst_bytes", "failed_logins", "connection_count"
]

security_classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-1",
    device=-1
)

threat_labels = [
    "a brute-force attack using repeated failed login attempts to guess passwords",
    "network port scanning to discover open ports on a server",
    "a denial-of-service attack overwhelming a server with excessive requests",
    "normal network activity with legitimate user login behavior"
]

security_docs = [
    "Brute-force attacks involve repeated login attempts to guess a user's password. Multiple failed logins from one IP address may indicate this activity.",
    "Port scanning identifies open ports and services on a network. Unusual connection attempts across many ports may indicate reconnaissance.",
    "Denial-of-service attacks overwhelm a server with excessive requests, making services unavailable to legitimate users.",
    "Normal network activity includes legitimate user logins and expected data transfers.",
    "Repeated authentication failures should be investigated using timestamps, source IP addresses, account activity, and authentication logs.",
    "Network monitoring can identify unusual connection counts and unexpected traffic patterns.",
    "Security alerts are indicators for investigation, not proof of malicious activity."
]

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

document_embeddings = embedding_model.encode(
    security_docs,
    convert_to_numpy=True
)

vector_db = faiss.IndexFlatL2(document_embeddings.shape[1])
vector_db.add(document_embeddings.astype("float32"))


def analyze_security_event(event, features):

    features["protocol"] = encoder.transform(
        [features["protocol"]]
    )[0]

    input_data = pd.DataFrame(
        [features],
        columns=feature_columns
    )

    ml_prediction = best_model.predict(input_data)[0]

    nlp_result = security_classifier(
        event,
        candidate_labels=threat_labels,
        hypothesis_template="This cybersecurity event is an example of {}."
    )

    query_embedding = embedding_model.encode(
        [event],
        convert_to_numpy=True
    )

    distances, indices = vector_db.search(
        query_embedding.astype("float32"), 3
    )

    intelligence = [
        security_docs[i] for i in indices[0] if i != -1
    ]

    return {
        "ml_detection": "Suspicious" if ml_prediction == 1 else "Normal",
        "nlp_classification": nlp_result["labels"][0],
        "nlp_score": round(float(nlp_result["scores"][0]) * 100, 2),
        "security_intelligence": intelligence
    }
'''

with open("../src/detection/inference.py", "w", encoding="utf-8") as file:
    file.write(inference_code)

print("CyberShield AI inference module created successfully!")

CyberShield AI inference module created successfully!


In [42]:
api_code = '''
import sys
from pathlib import Path

from fastapi import FastAPI
from pydantic import BaseModel

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))

from src.detection.inference import analyze_security_event

app = FastAPI(
    title="CyberShield AI",
    description="AI-powered cybersecurity threat analysis",
    version="1.0"
)

class SecurityEvent(BaseModel):
    event: str
    duration: int
    protocol: str
    src_bytes: int
    dst_bytes: int
    failed_logins: int
    connection_count: int

@app.get("/")
def home():
    return {
        "project": "CyberShield AI",
        "status": "API is running"
    }

@app.post("/analyze")
def analyze_event(data: SecurityEvent):

    features = {
        "duration": data.duration,
        "protocol": data.protocol,
        "src_bytes": data.src_bytes,
        "dst_bytes": data.dst_bytes,
        "failed_logins": data.failed_logins,
        "connection_count": data.connection_count
    }

    result = analyze_security_event(data.event, features)

    return {
        "project": "CyberShield AI",
        "event": data.event,
        "analysis": result
    }
'''

with open("../app/api/main.py", "w", encoding="utf-8") as file:
    file.write(api_code)

print("FastAPI connected to ML + NLP + RAG!")

FastAPI connected to ML + NLP + RAG!


In [45]:
dashboard_code = '''
import streamlit as st
import requests

API_URL = "http://127.0.0.1:8000"

st.set_page_config(
    page_title="CyberShield AI",
    page_icon="🛡️",
    layout="wide"
)

st.title("🛡️ CyberShield AI")
st.caption("AI-Powered Cybersecurity Threat Detection & Intelligence")

try:
    response = requests.get(API_URL, timeout=5)
    api_status = response.status_code == 200
except requests.RequestException:
    api_status = False

st.sidebar.title("System Status")

if api_status:
    st.sidebar.success("API Connected")
else:
    st.sidebar.error("API Offline")

st.sidebar.info("ML + NLP + RAG")

st.subheader("Security Event Analysis")

event = st.text_area(
    "Security Event Description",
    "Multiple failed login attempts detected from one IP address."
)

col1, col2, col3 = st.columns(3)

with col1:
    duration = st.number_input("Duration", min_value=0, value=250)
    protocol = st.selectbox("Protocol", ["TCP", "UDP", "ICMP"])

with col2:
    src_bytes = st.number_input("Source Bytes", min_value=0, value=500)
    dst_bytes = st.number_input("Destination Bytes", min_value=0, value=200)

with col3:
    failed_logins = st.number_input("Failed Logins", min_value=0, value=9)
    connection_count = st.number_input("Connection Count", min_value=0, value=15)

if st.button("Analyze Security Event", type="primary"):

    payload = {
        "event": event,
        "duration": int(duration),
        "protocol": protocol,
        "src_bytes": int(src_bytes),
        "dst_bytes": int(dst_bytes),
        "failed_logins": int(failed_logins),
        "connection_count": int(connection_count)
    }

    try:
        with st.spinner("Analyzing security event..."):
            result = requests.post(
                f"{API_URL}/analyze",
                json=payload,
                timeout=180
            )

        if result.status_code == 200:
            analysis = result.json()["analysis"]

            st.success("Analysis completed!")

            a, b, c = st.columns(3)
            a.metric("ML Detection", analysis["ml_detection"])
            b.metric("Threat Category", analysis["nlp_classification"])
            c.metric("NLP Score", f'{analysis["nlp_score"]}%')

            st.subheader("Security Intelligence")

            for i, document in enumerate(
                analysis["security_intelligence"], 1
            ):
                st.write(f"**{i}.** {document}")

            st.caption("AI predictions require analyst verification.")

        else:
            st.error(f"API Error: {result.status_code}")
            st.write(result.text)

    except requests.RequestException as error:
        st.error(f"Connection error: {error}")

st.divider()
st.caption("CyberShield AI | AI/ML Academic Project | Simulated Events")
'''

with open("../app/dashboard/dashboard.py", "w", encoding="utf-8") as file:
    file.write(dashboard_code)

print("Dashboard fixed and saved!")

Dashboard fixed and saved!


In [46]:
readme = """
# CyberShield AI

AI-powered cybersecurity threat detection and security intelligence platform.

## Features
- Machine Learning-based suspicious event detection.
- Hugging Face zero-shot threat classification.
- FAISS-based security knowledge retrieval.
- Retrieval-Augmented Generation (RAG).
- FastAPI backend.
- Streamlit interactive dashboard.
- Simulated cybersecurity event analysis.

## Technology Stack
Python, Pandas, NumPy, Scikit-learn, PyTorch,
Hugging Face Transformers, Sentence Transformers,
FAISS, FastAPI, Streamlit.

## Project Structure
- data/: Dataset storage
- models/: Trained ML models
- notebooks/: Development notebooks
- src/: Detection and intelligence modules
- app/api/: FastAPI backend
- app/dashboard/: Streamlit dashboard
- tests/: Testing
- docs/: Documentation

## Run Locally

Activate the environment:

    conda activate cybershield

Start the API from the project root:

    uvicorn app.api.main:app --reload

Start the dashboard in a second terminal:

    streamlit run app/dashboard/dashboard.py

API documentation:
http://127.0.0.1:8000/docs

Dashboard:
http://localhost:8501

## Important Limitations
This academic project currently uses simulated security events.
Its model scores are not calibrated probabilities.
Predictions are indicators for investigation, not proof of an attack.
Security intelligence should be verified by a human analyst.

## Project Status
Core ML, NLP, RAG, API and dashboard integration completed.
"""

with open("../README.md", "w", encoding="utf-8") as f:
    f.write(readme)

with open("../requirements.txt", "w", encoding="utf-8") as f:
    f.write("""
numpy
pandas
scikit-learn
matplotlib
seaborn
torch
transformers
datasets
sentence-transformers
faiss-cpu
fastapi
uvicorn
streamlit
requests
joblib
""".strip())

print("README and requirements updated successfully!")

README and requirements updated successfully!
